In [ ]:
# -*- coding: utf-8 -*-

tarea2_pi_JavieraManzo.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1mMICY6pDXpH-YkyMrGVwXiXA0hv1EUyw

# Tarea 2 - Procesamiento de Imágenes
# Alumno: Javiera Manzo González
# Profesor: Miguel Carrasco

###1. Utilice la imagen de soldadura que se encuentra en la intranet (adjunta con la tarea)

En esta oportunidad se realizará el procesamiento de una imagen proporcionada por el docente. Dicha imagen, llamada "fallas", corresponde a un registro de fallas presentes en una soldadura industrial, las cuales deben ser identificadas y corregidas. La imagen se encuentra en formato .tif (Tagged Image File Format), el cual permite almacenar imágenes de alta calidad sin compresión con pérdida, lo que asegura la fidelidad de la información visual.

Durante esta actividad se utilizan las mismas bibliotecas empleadas en la tarea 1: OpenCV, utilizada para la visualización y procesamiento de imágenes; Matplotlib, que permite graficar y representar visualmente los resultados; y NumPy, empleada para realizar los cálculos matemáticos necesarios en el procesamiento digital de imágenes.

*Referencia: https://www.adobe.com/es/creativecloud/file-types/image/raster/tiff-file.html*

In [ ]:
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('fallas.tif')

plt.imshow(img)
plt.title('Imagen original')
plt.show()


print("Dimensiones (alto, ancho, canales):", img.shape)

###2. Genere una nueva imagen ecualizada y comente los resultados respecto a la imagen original.

Luego de subir la imagen, se convierte a escala de grises utilizando la función "cvtColor" junto con el parámetro "COLOR_BGR2GRAY", ya que la función de ecualización de histograma solo puede aplicarse a imágenes de un solo canal y en escala de grises.

Finalmente, se aplica la función de ecualización "equalizeHist" y se muestra en pantalla el resultado obtenido y la imagen sin ecualizar.

In [ ]:
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('fallas.tif')

img_gray= cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

img_ecualizada = cv2.equalizeHist(img_gray)

plt.imshow(img, cmap='gray')
plt.title('Imagen original')
plt.show()
plt.imshow(img_ecualizada, cmap='gray')
plt.title('Imagen Ecualizada')
plt.show()

Realizando una comparación con la imagen original antes de aplicar la ecualización del histograma, se pueden notar a simple vista ciertas diferencias. La imagen ecualizada se ve ligeramente más clara y presenta un mayor contraste, lo que permite distinguir con mayor facilidad detalles que antes estaban atenuados y menos perceptibles, como la fallas,grietas e imperfecciones presentes en la soldadura.

###3. Genere una imagen “libre de ruido” empleando el filtro de la mediana con la imagen obtenida en paso-2. Busque un valor óptimo que atenúe completamente las fallas empleando el filtro de la mediana. Una vez definido un determinado valor, utilícelo en los siguientes pasos (Importante: el tamaño de máscara debe ser impar, ver Hint)

Para este ítem se generaron tres imágenes "libres de ruido" aplicando el filtro de la mediana, con el objetivo de observar de forma progresiva los cambios producidos y así identificar un rango de valores óptimos. Esto permitió determinar cuál es el tamaño de máscara más adecuado para atenuar las fallas sin perder detalles importantes de la imagen.

In [ ]:
import cv2
import matplotlib.pyplot as plt

img_median = cv2.medianBlur(img_ecualizada,61)
img_median2 = cv2.medianBlur(img_ecualizada,71)
img_median3 = cv2.medianBlur(img_ecualizada,81)


plt.imshow(img_median, cmap="gray")
plt.title('Imagen filtro mediana 61')
plt.show()
plt.imshow(img_median2, cmap="gray")
plt.title('Imagen filtro mediana 71')
plt.show()
plt.imshow(img_median3, cmap="gray")
plt.title('Imagen filtro mediana 81')
plt.show()

Se fueron probando distintos valores de tamaño de máscara. Podemos apreciar que el filtro de mediana con máscara de tamaño 61 aún presenta algunas manchas oscuras bastante visibles, por lo que es necesario utilizar un valor de máscara superior. Con una máscara de tamaño 71 ya se obtiene un resultado suficientemente limpio; aunque todavía se perciben algunas sombras grises, estas no representan un gran problema, por lo que este valor se considera óptimo. Al aplicar una máscara de tamaño 81, el resultado es muy similar al del tamaño 71 y cuesta notar diferencias significativas.

###4. Reste la imagen del paso-3 con la del paso-2 y despliegue el resultado por el browser (en colab). Analice en qué casos las fallas son más visibles según el parámetro del paso-3 y comente sus resultados. Solo las fallas debieran ser visibles como resultado de este paso.

Para realizar la resta entre la imagen obtenida en el paso 3 (imagen con filtro de la mediana) y la del paso 2 (imagen ecualizada), se utilizó la misma función empleada en la tarea 1 "absdiff" la cual permite calcular la diferencia absoluta entre ambas imágenes píxel a píxel.

*Referencia: https://omes-va.com/adicion-sustraccion-de-imagenes/*

In [ ]:
dif_3_2=cv2.absdiff(img_median2, img_ecualizada)

plt.imshow(dif_3_2,cmap='gray')
plt.title('Diferencia')
plt.show()

Luego de realizar la resta entre la imagen del paso 3 (filtrada con mediana) y la del paso 2 (ecualizada), se obtiene una imagen en la que podemos apreciar principalmente las grietas, imperfecciones y manchas, es decir, las fallas presentes en la soldadura. Esto es posible gracias a que la imagen filtrada representa cómo debería verse la soldadura sin defectos, por lo que al restarla de la imagen ecualizada, se destacan todos aquellos elementos que no deberían estar presentes.

Al utilizar una máscara intermedia de tamaño 71, se logra un resultado equilibrado: las fallas se ven más claras y aparecen rodeadas por un leve halo que las hace resaltar aún más. En cambio, con una máscara de tamaño 81, el halo se vuelve más visible. Por otro lado, con una máscara más pequeña, como la de 61, las fallas aún son visibles, pero tienden a verse más oscuras y no tan definidas.

###5. Binarice la imagen del paso 4 empleando una operación lógica según un umbral que usted defina. El objetivo de la binarización es que las fallas queden reflejadas en color blanco (valor 1) , y el fondo en color negro (valor 0).

Para este ítem se fue ajustando el valor del umbral con el fin de determinar con cuál se lograban apreciar mejor las fallas. Con un umbral inferior a 25, las imperfecciones se veían demasiado grandes y tendían a formar una única mancha extensa, perdiendo detalle. A medida que se aumentaba el umbral, las manchas iban disminuyendo de tamaño y separándose entre sí, lo que permitía distinguir mejor las diferentes fallas. Finalmente, se decidió establecer el umbral en 25, ya que con valores superiores las manchas más pequeñas y las grietas comenzaban a desaparecer, dejando visibles solo los puntos de mayor tamaño.

In [ ]:
J= dif_3_2 #resultado del paso 4
umbral=25
dst, bin = cv2.threshold(J,umbral,1,cv2.THRESH_BINARY)

plt.imshow(bin,cmap='gray')
plt.title('Imagen binarizada')
plt.show()

6. Finalmente multiplique punto a punto la imagen binaria del paso-5 con la imagen del paso-2 de tal modo que sólo sean visibles los niveles de intensidad de las fallas (en escala de grises), y el fondo se vea en color negro.

In [ ]:
paso5=bin #resultado del paso 5
paso2=img_ecualizada #resultado del paso 2

mult=cv2.multiply(paso5,paso2)

plt.imshow(mult,cmap='gray')
plt.title('resultado multiplicación paso 5 por paso 2')
plt.show()
plt.imshow(img,cmap='gray')
plt.title('img original')
plt.show()

Comparando el resultado de la multiplicación con la imagen original, podemos concluir que el objetivo de aislar las fallas, grietas e imperfecciones fue cumplido casi en su totalidad. Si bien algunas grietas menores no se logran apreciar completamente, la mayoría de las fallas significativas están presentes y se visualizan con claridad. Además, estas imperfecciones se representan en escala de grises, reflejando su intensidad, lo que permite identificarlas y evaluar su magnitud.

En general, la experiencia fue satisfactoria, permitiendo aplicar y poner en práctica los códigos estudiados en clases. A lo largo de cada paso se pudo experimentar con distintos parámetros, lo que exigió no solo el uso del conocimiento técnico, sino también del criterio personal para establecer valores óptimos que permitieran obtener resultados adecuados y coherentes con el objetivo de la tarea.